# Retrain the tuned sub-models and every blend (settings of the 14 Nov sweeps)

**Run All** together with `retrain_bayesian_fms.ipynb` (the two Bayesian FMs run on one CPU core in parallel). This notebook trains the
GPU / fast sub-models, then the blends - every sub-model is in the blend (all flags in `lib/config.py` are `True`) - and writes
`output/task_{1..5}_predictions.csv`. The Task 3 / Task 1 blends wait until `retrain_bayesian_fms.ipynb` has stored its model.

| task | retrained here | kept (trained 14 Nov with the current settings) |
|---|---|---|
| 2, 4, 5 | every sub-model (cheap ones too, so all are built with today's code) | - |
| 3 | every sub-model except the time-aware Bayesian FM (other notebook) | - |
| 1 | integrated model, I-AutoRec (Bayesian FM: other notebook) | bias model, SVD, SVD++, item kNN, zero model, FM, transformer |

Same calls as the `taskN_ensemble.ipynb` notebooks; a failing sub-model is logged and skipped (the blend uses its previously stored
version, or leaves it out if there is none). Log: `trained_models/retrain_log.txt`. Expected time: ~2.5 h (Task 5 and the Task 3 / 1
models are the long parts; the Bayesian FMs take ~70 min each with their refit).

In [ ]:
# no autoreload: the run uses the code and lib/config.py as they are now
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path
sys.path.append(str(Path.cwd()))               # assignment1.py and lib/ live next to this notebook (in submission/)

import numpy as np
import torch

import assignment1
import lib.config as model_configs
from lib.config import TRAINED_MODELS_DIR
from lib.helper_functioins import mse
from lib.model_storage import load_model_result, save_model_result, trained_model_path
from lib.models.blender import store_blender
from lib.models.pair_blender import store_pair_blender

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

LOG_PATH = TRAINED_MODELS_DIR / "retrain_log.txt"
run_start = time.time()                         # stored files newer than this were written by this run


def write_log(line: str):
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} {line}\n")
    print(line, flush=True)


# keeps Windows from sleeping while the notebook runs (released in the last cell)
def keep_computer_awake(enabled: bool):
    if sys.platform == "win32":
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001 if enabled else 0x80000000)


# trains one sub-model and stores it under its model name in trained_models/<task>/; a failure is logged and skipped
# (the blend then uses the version stored before, if there is one)
def train_and_store(task_name: str, task, model_name: str, trainer) -> bool:
    write_log(f"start    {task_name} {model_name}")
    start = time.perf_counter()
    try:
        result = trainer(task)
    except Exception as error:
        traceback.print_exc()
        write_log(f"FAILED   {task_name} {model_name}: {type(error).__name__}: {error}"[:400])
        return False
    results = result if isinstance(result, dict) else {model_name: result}
    for name, model_result in results.items():
        save_model_result(model_result, task_name, name)
        score = (f" | valid MSE {mse(model_result.valid_predictions, task.validation_data.ratings):.5f}"
                 if hasattr(task, "validation_data") and "zero_model" not in name else "")   # zero models predict P(rating 0)
        write_log(f"stored   {task_name} {name} ({(time.perf_counter() - start) / 60:.1f} min){score}")
    torch.cuda.empty_cache()
    return True


keep_computer_awake(True)
write_log(f"run started ({Path.cwd().name})")

In [ ]:
from dataclasses import replace
from lib.models.blender import train_blender
from lib.models.pair_blender import train_pair_blender


# stored sub-models of a blend; a missing one is left out (printed)
def stored_blend_inputs(task_name: str, model_names) -> dict:
    stored = [name for name in model_names if trained_model_path(task_name, name).exists()]
    missing = [name for name in model_names if name not in stored]
    if missing:
        write_log(f"NOTE     {task_name}: not stored -> left out of the blend: {', '.join(missing)}")
    return {name: load_model_result(task_name, name) for name in stored}


# waits until the other notebook has stored model_name in this run (file newer than run_start) or logged it as failed
def wait_for_stored_model(task_name: str, model_name: str, other_log_name: str, max_hours: float = 5.0):
    path, other_log = trained_model_path(task_name, model_name), TRAINED_MODELS_DIR / other_log_name
    write_log(f"waiting  for {task_name} {model_name} from retrain_bayesian_fms.ipynb")
    deadline = time.time() + max_hours * 3600
    while time.time() < deadline:
        if path.exists() and path.stat().st_mtime > run_start:
            write_log(f"found    {task_name} {model_name}")
            return
        if other_log.exists() and f"FAILED   {task_name} {model_name}" in other_log.read_text(encoding="utf-8"):
            write_log(f"NOTE     {task_name} {model_name} failed in the other notebook -> its older version (if any) is blended")
            return
        time.sleep(60)
    write_log(f"NOTE     gave up waiting for {task_name} {model_name} -> its older version (if any) is blended")


def pair_accuracy(task, blend_result) -> float:
    return float(np.mean((blend_result.valid_predictions > 0.5) == task.validation_pairs.first_is_positive))


# pair blend (Tasks 2 / 4 / 5) of every stored sub-model the config lists -> stored + prediction file
def blend_pair_task(task_name: str, task, blender_config, write_predictions, prediction_file):
    try:
        inputs = stored_blend_inputs(task_name, blender_config.combined_models)
        blend = train_pair_blender(task, inputs, replace(blender_config, combined_models=tuple(inputs)))
        store_pair_blender(blend, task_name)
        write_predictions(blend.test_predictions)
        write_log(f"blend    {task_name} pair blender ({len(inputs)} models) | out-of-fold accuracy {pair_accuracy(task, blend):.4f}"
                  f" | wrote {prediction_file.name}")
    except Exception as error:
        traceback.print_exc()
        write_log(f"FAILED   {task_name} blend: {type(error).__name__}: {error}"[:400])


# ridge + LightGBM blends (Tasks 1 / 3) of every stored sub-model the submitted blender lists -> both stored,
# the submitted method writes the prediction file
def blend_rating_task(task_name: str, task, blender_configs, submitted_method: str, write_predictions, prediction_file):
    inputs = stored_blend_inputs(task_name, blender_configs[0].blended_models)
    for blender_config in blender_configs:
        try:
            blend = train_blender(task, inputs, replace(blender_config, blended_models=tuple(inputs)))
            store_blender(blend, task_name)
            write_log(f"blend    {task_name} {blender_config.method} ({len(inputs)} models) | out-of-fold MSE "
                      f"{mse(blend.valid_predictions, task.validation_data.ratings):.5f}")
            if blender_config.method == submitted_method:
                write_predictions(blend.test_predictions)
                write_log(f"wrote    {prediction_file.name} ({blender_config.method} = the submitted blender)")
        except Exception as error:
            traceback.print_exc()
            write_log(f"FAILED   {task_name} {blender_config.method} blend: {type(error).__name__}: {error}"[:400])

## Task 2 - every sub-model, the pair blender, the prediction file

In [ ]:
task2 = assignment1.load_task2()
for model_name, trainer in assignment1.TASK2_SUB_MODEL_TRAINERS.items():
    train_and_store("task2", task2, model_name, trainer)

blend_pair_task("task2", task2, model_configs.TASK2_PAIR_BLENDER, assignment1.write_task2_predictions,
                assignment1.TASK2_PREDICTION_FILE)
del task2

## Task 4

In [ ]:
task4 = assignment1.load_task4()
for model_name, trainer in assignment1.TASK4_SUB_MODEL_TRAINERS.items():
    train_and_store("task4", task4, model_name, trainer)

blend_pair_task("task4", task4, model_configs.TASK4_PAIR_BLENDER, assignment1.write_task4_predictions,
                assignment1.TASK4_PREDICTION_FILE)
del task4

## Task 5 (EASE and time-weighted EASE come from one fit)

In [ ]:
task5 = assignment1.load_task5()
task5_trainers = {
    "popularity": assignment1.train_task5_popularity_model,
    "item_cosine": assignment1.train_task5_item_cosine_model,
    "transition": assignment1.train_task5_transition_model,
    "ease + time_ease": lambda task: dict(zip(("ease", "time_ease"), assignment1.train_task5_ease_models(task))),
    "mult_vae": assignment1.train_task5_mult_vae,
    "sasrec": assignment1.train_task5_sasrec,
}
for model_name, trainer in task5_trainers.items():
    train_and_store("task5", task5, model_name, trainer)

blend_pair_task("task5", task5, model_configs.TASK5_PAIR_BLENDER, assignment1.write_task5_predictions,
                assignment1.TASK5_PREDICTION_FILE)
del task5

## Task 3 - every sub-model except the time-aware Bayesian FM, then the blends (after the Bayesian FM is stored)

In [ ]:
task3 = assignment1.load_task3()
for model_name, trainer in assignment1.TASK3_SUB_MODEL_TRAINERS.items():
    if model_name != "time_bayesian_factorization_machine":            # retrain_bayesian_fms.ipynb
        train_and_store("task3", task3, model_name, trainer)

wait_for_stored_model("task3", "time_bayesian_factorization_machine", "retrain_bayesian_fm_log.txt")
blend_rating_task("task3", task3, (model_configs.TASK3_RIDGE_BLENDER, model_configs.TASK3_LIGHTGBM_BLENDER),
                  model_configs.TASK3_SUBMITTED_BLENDER.method, assignment1.write_task3_predictions,
                  assignment1.TASK3_PREDICTION_FILE)

## Task 1 - integrated model and I-AutoRec, then the blends (after the Bayesian FM is stored)

In [ ]:
task1 = assignment1.load_task1()
for model_name in ("integrated_model", "autorec"):
    train_and_store("task1", task1, model_name, assignment1.TASK1_SUB_MODEL_TRAINERS[model_name])

wait_for_stored_model("task1", "bayesian_factorization_machine", "retrain_bayesian_fm_log.txt")
blend_rating_task("task1", task1, (model_configs.TASK1_RIDGE_BLENDER, model_configs.TASK1_LIGHTGBM_BLENDER),
                  model_configs.TASK1_SUBMITTED_BLENDER.method, assignment1.write_task1_predictions,
                  assignment1.TASK1_PREDICTION_FILE)

In [ ]:
keep_computer_awake(False)
write_log("run finished")
print(LOG_PATH.read_text(encoding="utf-8"))